# 02 - Transform

In [ ]:
# Databricks Notebook: 02-transform
# Purpose: Transform Bronze → Silver using validation + conforming logic

from pyspark.sql.functions import col, to_date, when


In [ ]:

# ---------------------------
# CONFIG
# ---------------------------
catalog = "main"
bronze_schema = "bronze"
silver_schema = "silver"

bronze_table = f"{catalog}.{bronze_schema}.events_bronze"
silver_table = f"{catalog}.{silver_schema}.events_silver"


In [ ]:

# ---------------------------
# READ BRONZE
# ---------------------------

df = spark.table(bronze_table)


In [ ]:

# ---------------------------
# BASIC DATA QUALITY CHECKS
# ---------------------------

df_clean = (
    df.dropDuplicates()
      .filter(col("eventId").isNotNull())
      .filter(col("timestamp").isNotNull())
)


In [ ]:

# ---------------------------
# TRANSFORM & CONFORM DATA
# ---------------------------

df_transformed = (
    df_clean.withColumn("event_date", to_date(col("timestamp")))
            .withColumn("event_type",
                when(col("type").isNull(), "unknown")
                .otherwise(col("type"))
            )
)


In [ ]:

# ---------------------------
# WRITE TO SILVER
# ---------------------------

df_transformed.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table)

display(df_transformed)